In [ ]:
from keras.models import load_model
from collections import deque
import matplotlib.pyplot as plt
import numpy as np
import cv2
import telepot
import geopy.geocoders
from datetime import datetime
import pytz

# ─── CONFIGURATION ────────────────────────────────────────────────────────────
BOT_TOKEN = "YOUR_BOT_TOKEN_HERE"       # Replace with your Telegram bot token
CHAT_ID   = YOUR_CHAT_ID_HERE           # Replace with your Telegram group chat ID
LOCATION  = "YOUR_LOCATION_HERE"        # Replace e.g. "B.M.S. College, Bangalore\n 12.9410° N, 77.5655° E"
# ──────────────────────────────────────────────────────────────────────────────

def getTime():
    IST = pytz.timezone('Asia/Kolkata')
    timeNow = datetime.now(IST)
    return timeNow

def print_results(video=None, webcam=False):
    trueCount = 0
    imageSaved = 0
    filename = 'savedImage.jpg'

    print("Loading model ...")
    model = load_model('modelnew.h5')
    Q = deque(maxlen=128)

    if webcam:
        cap = cv2.VideoCapture(0)
    else:
        cap = cv2.VideoCapture(video)

    writer = None
    (W, H) = (None, None)

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        if W is None or H is None:
            (H, W) = frame.shape[:2]

        frame_resized = cv2.resize(frame, (128, 128))
        output = frame_resized.copy()

        preds = model.predict(np.expand_dims(frame_resized, axis=0))[0]
        Q.append(preds)

        results = np.array(Q).mean(axis=0)
        i = (preds > 0.50)[0]
        label = i

        text_color = (0, 255, 0)

        if label:
            text_color = (0, 0, 255)
            trueCount = trueCount + 1

        text = "Violence: {}".format(label)
        FONT = cv2.FONT_HERSHEY_SIMPLEX

        cv2.putText(output, text, (5, 15), FONT, 0.35, text_color, 1)

        if writer is None and not webcam:
            fourcc = cv2.VideoWriter_fourcc(*"MJPG")
            writer = cv2.VideoWriter("recordedVideo.avi", fourcc, 30, (W, H), True)

        if not webcam:
            writer.write(output)
            cv2.imshow('Frame', output)
        else:
            resized_output = cv2.resize(output, (1280, 720))
            cv2.imshow('Webcam', resized_output)

        if trueCount == 50:
            if imageSaved == 0:
                if label:
                    cv2.imwrite(filename, output)
                    imageSaved = 1

            timeMoment = getTime()
            bot = telepot.Bot(BOT_TOKEN)
            bot.sendMessage(CHAT_ID, f"VIOLENCE ALERT!! \nLOCATION: {LOCATION} \nTIME: {timeMoment}")
            bot.sendPhoto(CHAT_ID, photo=open(filename, 'rb'))

            trueCount = 0

        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            break

    print("[INFO] cleaning up...")
    if writer is not None and not webcam:
        writer.release()
    cap.release()
    cv2.destroyAllWindows()

# Call the function to start testing the webcam
print_results(webcam=True)